# DATA 612 — Benchmark data prep: Retailrocket (12h windows)

Converts the Retailrocket e-commerce log into **exactly the same tables, labels and splits as the Criteo
pipeline** (`src/data/preprocess.py`, `src/data/splits.py`, `scripts/prepare_criteo.py`), so every training
notebook runs on it unchanged — only `DATASET = "retailrocket_w12h"` changes.

**Mapping (Criteo → Retailrocket)**

| Criteo | Retailrocket |
|---|---|
| user (`uid`) | `visitorid` |
| impression row | any event (view, add-to-cart, transaction) |
| campaign (token) | item **category** (as of the event time), top `MAX_CATEGORIES` + `other` + `unknown` |
| click | add-to-cart or transaction |
| conversion (label) | **transaction** (deduplicated on `transactionid`) |
| cost | not available → 0 |

Same rules as Criteo: 12h windows, predict window t+1 from windows ≤ t, a visitor is a prediction position if
active in the last 7 days, chronological 70/15/15 split by label window, hashed 70/15/15 user groups.

**Runtime:** CPU is enough (High-RAM recommended). About 2.8M events, so this is much smaller than Criteo.

## 1. Settings

In [ ]:
import os, sys, json, time, shutil, subprocess, importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/DATA612_data")
    REPO_DIR  = Path("/content/repo")
    WORK_DIR  = Path("/content/work")
else:
    DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", "drive_sim/DATA612_data")).resolve()
    REPO_DIR  = Path(os.environ.get("REPO_DIR", "repo")).resolve()
    WORK_DIR  = Path(os.environ.get("WORK_DIR", "work")).resolve()

WINDOW_HOURS   = 12
MAX_CATEGORIES = 675        # same vocabulary size as Criteo's 675 campaigns; rarer categories -> "other"
N_SHARDS       = 4
RAW_DIR = DRIVE_DIR / "raw" / "retailrocket"
NAME    = f"retailrocket_w{WINDOW_HOURS:g}h"
RAW_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Raw files:", RAW_DIR, "| output name:", NAME)

## 2. Repo (the shared pipeline) and packages

In [ ]:
REPO_URL = "https://github.com/Abhyudaya01/criteo-customer-journey-transformers.git"
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False)
elif IN_COLAB:
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / "scripts"))

need = [p for p, m in {"pyarrow": "pyarrow", "kagglehub": "kagglehub"}.items() if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)

import numpy as np, pandas as pd, yaml
from src.data.preprocess import DataColumns, build_campaign_vocab, build_windowed_dataset, finalize_meta, \
    n_windows_for, write_table
from src.data.splits import window_boundaries
import prepare_criteo as pc          # reuse: add_split_columns, split_counts, merge_counts, shard_ids, TABLES
cfg = yaml.safe_load((REPO_DIR / "configs" / "base.yaml").read_text())
SEED = cfg["project"].get("seed", 42)
print("pandas", pd.__version__, "| seed", SEED)

## 3. Download (once, to Drive)
Uses `kagglehub`. If it asks for credentials: on kaggle.com → Settings → **Create New Token**, then in Colab
add two **Secrets** (key icon on the left): `KAGGLE_USERNAME` and `KAGGLE_KEY`, and re-run this cell.
Alternatively, download the zip from https://www.kaggle.com/datasets/retailrocket/ecommerce-dataset yourself,
unzip it into `MyDrive/DATA612_data/raw/retailrocket/`, and re-run.

In [ ]:
FILES = ["events.csv", "item_properties_part1.csv", "item_properties_part2.csv", "category_tree.csv"]
if not all((RAW_DIR / f).exists() for f in FILES):
    if IN_COLAB:
        try:
            from google.colab import userdata
            for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
                try:
                    os.environ[k] = userdata.get(k)
                except Exception:
                    pass
        except ImportError:
            pass
    import kagglehub
    src = Path(kagglehub.dataset_download("retailrocket/ecommerce-dataset"))
    for f in FILES:
        hit = next(src.rglob(f))
        shutil.copy2(hit, RAW_DIR / f)
missing = [f for f in FILES if not (RAW_DIR / f).exists()]
assert not missing, f"Missing raw files: {missing}"
for f in FILES:
    print(f"{f:<28} {(RAW_DIR / f).stat().st_size / 1e6:8.1f} MB")

## 4. Map Retailrocket events to the Criteo input schema

In [ ]:
t0 = time.time()
ev = pd.read_csv(RAW_DIR / "events.csv")
ev = ev.sort_values(["timestamp", "visitorid"], kind="mergesort").reset_index(drop=True)
print(f"events: {len(ev):,} | visitors: {ev['visitorid'].nunique():,} | items: {ev['itemid'].nunique():,} | "
      f"transactions: {ev['transactionid'].nunique():,}")
print("event types:", ev["event"].value_counts().to_dict())
print("period:", pd.to_datetime(ev["timestamp"].min(), unit="ms"), "->", pd.to_datetime(ev["timestamp"].max(), unit="ms"))

# Item category as of the event time (item properties are time-stamped snapshots).
props = pd.concat([pd.read_csv(RAW_DIR / f) for f in FILES[1:3]], ignore_index=True)
cat = props.loc[props["property"] == "categoryid", ["timestamp", "itemid", "value"]].copy()
cat["categoryid"] = pd.to_numeric(cat["value"], errors="coerce").astype("Int64")
cat = cat.dropna(subset=["categoryid"]).drop(columns="value").sort_values("timestamp")
del props

left = ev[["timestamp", "itemid"]].reset_index().sort_values("timestamp")
asof = pd.merge_asof(left, cat, on="timestamp", by="itemid", direction="backward")
first_cat = cat.sort_values("timestamp").drop_duplicates("itemid").set_index("itemid")["categoryid"]
asof["categoryid"] = asof["categoryid"].fillna(asof["itemid"].map(first_cat))   # snapshot before first property row
asof = asof.sort_values("index")
category = asof["categoryid"].fillna(-1).astype("int64").to_numpy()            # -1 = unknown category
print(f"events with a known category: {(category >= 0).mean():.1%}")

# Keep the MAX_CATEGORIES - 2 most frequent categories; the rest -> "other" (-2). Plus "unknown" (-1).
counts = pd.Series(category[category >= 0]).value_counts()
keep = set(counts.index[:MAX_CATEGORIES - 2])
token = np.where(category < 0, -1, np.where(np.isin(category, list(keep)), category, -2))
print(f"distinct categories: {len(counts):,} -> tokens: {len(np.unique(token))} "
      f"(share of events in 'other': {(token == -2).mean():.1%})")

is_tx = (ev["event"] == "transaction").to_numpy()
ts = ((ev["timestamp"] - ev["timestamp"].min()) // 1000).to_numpy(np.int64)    # seconds from the first event
raw = pd.DataFrame({
    "timestamp": ts,
    "uid": ev["visitorid"].to_numpy(np.int64),
    "campaign": token.astype(np.int64),
    "conversion": is_tx.astype(np.int8),
    "conversion_timestamp": np.where(is_tx, ts, -1).astype(np.int64),            # a purchase converts at its own time
    "conversion_id": np.where(is_tx, ev["transactionid"].fillna(-1), -1).astype(np.int64),
    "click": ev["event"].isin(["addtocart", "transaction"]).to_numpy().astype(np.int8),
    "cost": np.zeros(len(ev), dtype=np.float32),
})
del ev, asof, left
print(f"mapped to Criteo schema in {time.time() - t0:.0f}s:", raw.shape)
raw.head()

## 5. Build windows, labels and splits (same code path as `scripts/prepare_criteo.py`)

In [ ]:
cols = DataColumns.from_config(cfg["data"])
win = cfg["windowing"]
horizon = int(win.get("horizon_windows", 1))
max_inactive = win.get("max_inactive_hours", 168)
s_cfg = cfg["splits"]

vocab = build_campaign_vocab(raw, cols)
shards = pc.shard_ids(raw[cols.user], N_SHARDS, SEED)
n_windows = n_windows_for(raw, WINDOW_HOURS, cols)
bounds = window_boundaries(n_windows, s_cfg.get("train_frac", 0.70), s_cfg.get("val_frac", 0.15), horizon)
out_dir = WORK_DIR / "processed" / NAME
if out_dir.exists():
    shutil.rmtree(out_dir)
write_table(vocab, out_dir / "campaign_vocab.parquet")

t0 = time.time()
totals, counts, meta = {k: 0 for k in pc.SUMMED_META}, {}, {}
for i in range(N_SHARDS):
    part = raw[shards == i]
    if part.empty:
        continue
    res = build_windowed_dataset(part, WINDOW_HOURS, max_inactive, horizon, cols,
                                 vocab=vocab, n_windows=n_windows, timestamp_unit="seconds")
    res["positions"] = pc.add_split_columns(res["positions"], bounds, cfg, SEED)
    counts = pc.merge_counts(counts, pc.split_counts(res["positions"]))
    for t in pc.TABLES:
        write_table(res[t], out_dir / t / f"part-{i:03d}.parquet")
    meta = res["meta"]
    for k in pc.SUMMED_META:
        totals[k] += meta[k]
    print(f"  shard {i + 1}/{N_SHARDS}: {meta['n_positions']:,} positions")

meta.update(totals)
meta["n_campaigns"] = int(len(vocab))
meta = finalize_meta(meta)
meta["splits"] = {"boundaries": bounds.as_dict(), **counts}
meta.update({"seed": SEED, "n_shards": N_SHARDS, "source_dataset": "retailrocket",
             "campaign_token": f"item category (top {MAX_CATEGORIES - 2} + other + unknown)",
             "conversion": "transaction", "click": "addtocart or transaction", "cost": "not available (0)"})
(out_dir / "meta.json").write_text(json.dumps(meta, indent=2))
print(f"built in {time.time() - t0:.0f}s")

## 6. Sanity checks

In [ ]:
pos = pd.read_parquet(out_dir / "positions")
summary = {
    "window hours": WINDOW_HOURS, "windows": meta["n_windows"], "events": meta["n_impressions"],
    "visitors": meta["n_users"], "tokens (categories)": meta["n_campaigns"],
    "distinct transactions": meta["n_conversions_total"], "positions": meta["n_positions"],
    "positive positions": meta["n_positive_positions"],
    "positive rate": f"{meta['positive_rate_conversion']:.4%}",
    "positions with a next-window category": f"{meta['frac_positions_with_next_campaign']:.1%}",
    "conversion windows covered": f"{meta['frac_conversion_windows_covered']:.1%}",
}
display(pd.Series(summary, name=NAME).to_frame())
print("Split boundaries (label windows):", meta["splits"]["boundaries"])
display(pd.DataFrame(meta["splits"]["time_split"]).T.rename_axis("time_split"))
display(pd.DataFrame(meta["splits"]["user_group"]).T.rename_axis("user_group"))

checks = {
    "labels come from the next window": bool((pos["label_window"] == pos["window"] + horizon).all()),
    "no position beyond the last window": bool(pos["label_window"].max() <= meta["n_windows"] - 1),
    "positives exist in every split": all(meta["splits"]["time_split"][s]["n_positive"] > 0 for s in ("train", "val", "test")),
    "same columns as Criteo positions": set(pos.columns) >= {"uid", "window", "label_window", "is_active",
        "windows_since_active", "y_conversion", "n_next_campaigns", "time_split", "user_group"},
}
for k, v in checks.items():
    print(("OK      " if v else "PROBLEM ") + k)

## 7. Copy to Drive

In [ ]:
dst = DRIVE_DIR / "processed" / NAME
t0 = time.time()
shutil.copytree(out_dir, dst, dirs_exist_ok=True)
size = sum(f.stat().st_size for f in dst.rglob("*") if f.is_file()) / 1e9
print(f"{NAME} -> {dst} ({size:.2f} GB, {time.time() - t0:.0f}s)")
if IN_COLAB:
    drive.flush_and_unmount()
    print("Drive synced and unmounted.")
print('Train on it later with DATASET = "' + NAME + '" in any training notebook.')

## Notes for the report
- Retailrocket has **no ad spend or campaigns**; categories play the role of campaigns, and transactions play the
  role of conversions. It tests whether the models generalise to general user-behaviour data, not advertising.
- Item categories change over time in the raw data; each event uses the category **as of its own timestamp**.
- The dataset contains some bot-like visitors with very many events; they are kept as-is (no filtering), the
  same as Criteo.